In [1]:
import pandas as pd
import numpy as np
import torch

import matplotlib.pyplot as plt
import plotly.graph_objects as go

import urllib3 # to download data directly from web

# Download 65N July data to reproduce van Nes results

## Notes
- see [Readme file](https://www.ncei.noaa.gov/pub/data/paleo/climate_forcing/orbital_variations/insolation/readme_insolation.txt)
- spacing is already 1 kyr
- we take 800 BP to present
- we use the same dataset for Vostok and EPICA
- BP references the present as 1950.
- question maintains while 65N was chosen
- Check temporal direction: -400 BC: was 422.02 & 0 BC (now aka 1950 A.D.) is 426.76.

## File details from README
1. File ORBIT91: 0-5 Myr B.P.  
    . first column: time in kyr (negative for the past; origin (0) 
                                 is 1950 A.D.)  
    . second column: eccentricity, ECC   
    . third col: longitude of perihelion from moving vernal equinox 
      in degrees, OMEGA  
    . fourth column: obliquity in degree and decimals, OBL   
    . fifth column: climatic precession, ECC . SIN(OMEGA)   
    . **sixth column: mid-month insolation 65N for July in W/m^2**    
    . seventh column: mid-month insolation 65S for January in W/m^2   
    . eighth column: mid-month insolation 15N for July in W/m^2  
    . ninth column: mid-month insolation 15S for January in W/m^2   

In [2]:
# link to data file
url = "https://www.ncei.noaa.gov/pub/data/paleo/climate_forcing/orbital_variations/insolation/orbit91"

# use urllib3 package to download .dat data directly from web
# Creating a PoolManager instance for sending requests.
http = urllib3.PoolManager()

# Sending a GET request and getting back response as HTTPResponse object.
resp = http.request("GET", url)

# create new (empty) text file ('w' stands for write mode)
file = open('../data/insolation/orbit91.text', 'w')
# populate text file with our data
file.write(resp.data.decode('utf-8-sig'))
# close file (write mode)
file.close()

In [3]:
# open file in read mode
file = open('../data/insolation/orbit91.text', "r")

list = []

for index, line in enumerate(file):
    # skip headings and very old rows
    if (index > 2) & (index < 804):
        list.append(line.split())

In [4]:
insolation_65N_ts_all = np.array(list, dtype = float)

# select time and 65NJun for time series (6th column is index 5)
insolation_65N_ts = np.array(list, dtype = float)[:, [0, 5]]

# Export Vostok time interval

In [5]:
insol_timeseries_400kyr_vostok = torch.flip(torch.tensor(insolation_65N_ts[0:(400 + 1), 1]), dims = [0])

In [6]:
def affirm_uniqueness(ts):
    # Check for duplicates and add noise 
    dupes = ts.shape[0] - torch.unique(ts.to(torch.float32)).shape[0]
    print("There are ", dupes, " duplicates in the timeseries.")
    if dupes > 0:
        noise_level = 0.001
        while dupes > 0:
            ts = ts + torch.randn(ts.shape[0]) * noise_level
            # recalculate dupes
            dupes = ts.shape[0] - torch.unique(ts.to(torch.float32)).shape[0]
            print("Now we have ", dupes, " dupes.")
    return ts

In [7]:
insol_timeseries_400kyr_vostok = affirm_uniqueness(insol_timeseries_400kyr_vostok)

There are  10  duplicates in the timeseries.
Now we have  0  dupes.


In [8]:
torch.save(insol_timeseries_400kyr_vostok, "../data/vostok/insolation/insolation_65N_400kyr_timeseries.pt")

# Export EPICA time interval

In [9]:
insol_timeseries_800kyr_epica = torch.flip(torch.tensor(insolation_65N_ts[0:(800 + 1), 1]), dims = [0])

In [10]:
insol_timeseries_800kyr_epica = affirm_uniqueness(insol_timeseries_800kyr_epica)

There are  44  duplicates in the timeseries.
Now we have  1  dupes.
Now we have  2  dupes.
Now we have  0  dupes.


In [11]:
torch.save(insol_timeseries_800kyr_epica, "../data/epica/insolation/insolation_65N_800kyr_timeseries.pt")